# Output Parsers — Structured Data from LLMs

Turn raw LLM text into Python objects: dicts, lists, Pydantic models, enums, and more.

---

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic pydantic

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1 · StrOutputParser — The Default

Extracts the plain text string from an `AIMessage`. The simplest parser — use it when you just want the raw response.

In [ ]:
# Without parser — returns an AIMessage object
raw = llm.invoke("What is RLHF in one sentence?")
print(type(raw))   # <class 'langchain_core.messages.ai.AIMessage'>
print(raw.content) # the actual text is inside .content

In [ ]:
# With StrOutputParser — returns a clean string
chain = llm | StrOutputParser()
result = chain.invoke("What is RLHF in one sentence?")
print(type(result))  # <class 'str'>
print(result)

## 2 · JsonOutputParser — Parse JSON Responses

When you need a Python dict back from the LLM. Automatically injects format instructions into the prompt.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. {format_instructions}"),
    ("human", "Give me 3 facts about {topic}. Return as a JSON with keys: fact_1, fact_2, fact_3")
])

chain = prompt | llm | parser

result = chain.invoke({
    "topic": "black holes",
    "format_instructions": parser.get_format_instructions()
})

print(type(result))  # <class 'dict'>
print(result)

## 3 · PydanticOutputParser — Validated Structured Output

The most powerful parser. Define a Pydantic model, and the parser:
1. Generates format instructions from the schema
2. Parses the LLM output into a validated Pydantic object
3. Raises clear errors if validation fails

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define your desired output structure
class MovieReview(BaseModel):
    title: str = Field(description="Name of the movie")
    rating: float = Field(description="Rating out of 10")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    verdict: str = Field(description="One-line recommendation")

parser = PydanticOutputParser(pydantic_object=MovieReview)

# See what format instructions look like
print("=== Format Instructions ===")
print(parser.get_format_instructions())

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a film critic. {format_instructions}"),
    ("human", "Review the movie: {movie}")
])

chain = prompt | llm | parser

review = chain.invoke({
    "movie": "Inception",
    "format_instructions": parser.get_format_instructions()
})

# result is a validated Pydantic object
print(type(review))       # <class 'MovieReview'>
print(f"Title: {review.title}")
print(f"Rating: {review.rating}")
print(f"Pros: {review.pros}")
print(f"Cons: {review.cons}")
print(f"Verdict: {review.verdict}")

## 4 · CommaSeparatedListOutputParser — Quick Lists

For when you need a Python `list[str]` and don't need full JSON/Pydantic overhead.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_template(
    "List 5 popular {category}. {format_instructions}"
)

chain = prompt | llm | parser

result = chain.invoke({
    "category": "Python web frameworks",
    "format_instructions": parser.get_format_instructions()
})

print(type(result))  # <class 'list'>
print(result)        # ['Django', 'Flask', 'FastAPI', ...]

## 5 · StructuredOutputParser — Schema Without Pydantic

Define output structure with simple `ResponseSchema` objects. Lighter than Pydantic when you just need basic key-value output.

In [ ]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

schemas = [
    ResponseSchema(name="summary", description="A 2-sentence summary"),
    ResponseSchema(name="difficulty", description="beginner, intermediate, or advanced"),
    ResponseSchema(name="time_to_learn", description="Estimated hours to learn"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

prompt = ChatPromptTemplate.from_template(
    "Describe the programming language {language}.\n{format_instructions}"
)

chain = prompt | llm | parser

result = chain.invoke({
    "language": "Rust",
    "format_instructions": parser.get_format_instructions()
})

print(type(result))  # <class 'dict'>
print(result)

## 6 · Enum Output Parser — Constrained Choices

Force the LLM to pick from a fixed set of options. Great for classification tasks.

In [ ]:
from langchain.output_parsers import EnumOutputParser
from enum import Enum

class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"

parser = EnumOutputParser(enum=Sentiment)

prompt = ChatPromptTemplate.from_template(
    "Classify the sentiment of this text. Respond with ONLY one word.\n"
    "{format_instructions}\n\nText: {text}"
)

chain = prompt | llm | parser

result = chain.invoke({
    "text": "This product exceeded all my expectations!",
    "format_instructions": parser.get_format_instructions()
})

print(type(result))   # <enum 'Sentiment'>
print(result)          # Sentiment.POSITIVE
print(result.value)    # 'positive'

## 7 · OutputFixingParser — Auto-Repair Bad Output

Wraps another parser. If parsing fails, it sends the error + raw output back to the LLM to fix it automatically.

In [ ]:
from langchain.output_parsers import OutputFixingParser

# Base parser expects a Pydantic MovieReview
base_parser = PydanticOutputParser(pydantic_object=MovieReview)

# Wrap it — if base_parser fails, this retries with the LLM
fixing_parser = OutputFixingParser.from_llm(
    parser=base_parser,
    llm=llm
)

# Simulate malformed LLM output
bad_output = '{"title": "Inception", "rating": "eight"}'

# base_parser.parse(bad_output)  # This would FAIL — missing fields, wrong type

# fixing_parser sends the error back to the LLM to fix it
fixed = fixing_parser.parse(bad_output)
print(type(fixed))  # <class 'MovieReview'>
print(fixed)

## 8 · Practical Pattern — Pydantic Parser in a Real Chain

End-to-end example: extract structured data from unstructured text.

In [ ]:
class JobPosting(BaseModel):
    company: str = Field(description="Company name")
    role: str = Field(description="Job title")
    skills: List[str] = Field(description="Required skills")
    yoe: int = Field(description="Minimum years of experience")
    remote: bool = Field(description="Whether the job is remote")

parser = PydanticOutputParser(pydantic_object=JobPosting)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured job information from the text. {format_instructions}"),
    ("human", "{raw_posting}")
])

chain = prompt | llm | parser

raw_text = """
We're hiring at Acme Corp! Looking for a Senior ML Engineer with 5+ years of 
experience. Must know Python, PyTorch, and Kubernetes. This is a fully remote 
position with competitive salary and equity.
"""

job = chain.invoke({
    "raw_posting": raw_text,
    "format_instructions": parser.get_format_instructions()
})

print(f"Company:  {job.company}")
print(f"Role:     {job.role}")
print(f"Skills:   {job.skills}")
print(f"YOE:      {job.yoe}")
print(f"Remote:   {job.remote}")

## 9 · with_structured_output — The Modern Way (LangChain v0.2+)

The newest and simplest approach. Skips the parser entirely — bind Pydantic schema directly to the LLM.

In [ ]:
class TechSummary(BaseModel):
    name: str = Field(description="Technology name")
    category: str = Field(description="Category: language, framework, tool, or platform")
    best_for: str = Field(description="Primary use case in one sentence")
    popularity: int = Field(description="Popularity score 1-10")

# Bind Pydantic model directly to the LLM — no separate parser needed
structured_llm = llm.with_structured_output(TechSummary)

result = structured_llm.invoke("Tell me about FastAPI")

print(type(result))  # <class 'TechSummary'>
print(f"Name:       {result.name}")
print(f"Category:   {result.category}")
print(f"Best for:   {result.best_for}")
print(f"Popularity: {result.popularity}")

In [ ]:
# with_structured_output also works in chains
prompt = ChatPromptTemplate.from_template("Analyze the tech: {tech}")
chain = prompt | structured_llm

# Batch multiple
results = chain.batch([
    {"tech": "Docker"},
    {"tech": "Kubernetes"},
    {"tech": "Terraform"},
])

for r in results:
    print(f"{r.name} ({r.category}) — {r.best_for}")

---

## Key Takeaways

| Parser | Returns | Best For |
|--------|---------|----------|
| `StrOutputParser` | `str` | Default — raw text |
| `JsonOutputParser` | `dict` | Quick JSON extraction |
| `PydanticOutputParser` | Pydantic model | Validated structured data |
| `CommaSeparatedListOutputParser` | `list[str]` | Simple lists |
| `StructuredOutputParser` | `dict` | Schema without Pydantic |
| `EnumOutputParser` | `Enum` | Classification / fixed choices |
| `OutputFixingParser` | varies | Auto-repair failed parses |
| `with_structured_output()` | Pydantic model | Modern approach (v0.2+) |

**Next →** [04 · Document Loaders](../04-document-loaders/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*